<style>
h1, h1 *, 
h2, h2 *, 
h3, h3 *, 
h4, h4 *, 
h5, h5 *, 
h6, h6 * {
  font-family: "Times New Roman", Times, serif !important;
}
</style>

## **Optimising Marketing with Artificial Intelligence (OMAI)**

<style>
h1, h1 *, 
h2, h2 *, 
h3, h3 *, 
h4, h4 *, 
h5, h5 *, 
h6, h6 * {
  font-family: "Times New Roman", Times, serif !important;
}
</style>

### **Transformer-Based Sentiment Analysis Models - DistilBERT (Round 3)**

---

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

OMAI_distilbert_round3 = pd.read_csv('OMAI - Data (TBSAMs) (Round 3).csv')
OMAI_distilbert_round3 = OMAI_distilbert_round3.drop('Unnamed: 0', axis = 1)
OMAI_distilbert_round3.iloc[0:1]

In [ ]:
import os
current_folder = os.getcwd()
print(current_folder)

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import transformers
import os
import shutil
import tarfile
import pandas as pd
import re
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.offline as pyo
import plotly.graph_objects as go
from wordcloud import WordCloud, STOPWORDS
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from bs4 import BeautifulSoup
from transformers import BertTokenizer, TFBertForSequenceClassification

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
OMAI_distilbert_round3['sentiment_scoreR3'].value_counts(normalize = True)

sentiment_scoreR3
1    0.720865
0    0.279135
Name: proportion, dtype: float64

In [ ]:
from sklearn.model_selection import train_test_split

def data_splits13(OMAI_distilbert_round3): 
    train_df13, temp_df13 = train_test_split(OMAI_distilbert_round3, test_size = 0.2,
                                             stratify = OMAI_distilbert_round3['sentiment_scoreR3'], random_state = 1500)

    val_df13, test_df13 = train_test_split(temp_df13, test_size = 0.5, 
                                           stratify = temp_df13['sentiment_scoreR3'], random_state = 1500)

    return (train_df13.reset_index(drop = True), val_df13.reset_index(drop = True), test_df13.reset_index(drop = True))

train_df13, val_df13, test_df13 = data_splits13(OMAI_distilbert_round3)

train_text13 = train_df13['combined_text']
train_labels13 = train_df13['sentiment_scoreR3']

val_text13 = val_df13['combined_text']
val_labels13 = val_df13['sentiment_scoreR3']

test_text13 = test_df13['combined_text']
test_labels13 = test_df13['sentiment_scoreR3']

In [ ]:
from transformers import AutoModel, BertTokenizerFast

bert13 = AutoModel.from_pretrained('distilbert-base-cased')
tokenizer13 = BertTokenizerFast.from_pretrained('distilbert-base-cased')

In [ ]:
seq_len13 = [len(i.split()) for i in train_text13]
pd.Series(seq_len13).hist(bins = 20)

In [ ]:
import psutil
import os
import torch
import numpy as np
from sklearn.model_selection import train_test_split

def print_memory_usage13(note = ""):
    process = psutil.Process(os.getpid())
    mem = process.memory_info().rss / (1024 ** 3)
    print(f"[{note}] Memory usage: {mem:.2f} GB")

def batch_tokenize13(texts13, tokenizer13, max_length = 128, tokenize_batch_size13 = 10000):
    input_ids13, attention_masks13 = [], []
    print_memory_usage13("Start batch_tokenize13")

    for i in range(0, len(texts13), tokenize_batch_size13):
        print(f"  → Tokenizing batch {i}–{i + tokenize_batch_size13}")
        batch13 = tokenizer13.batch_encode_plus(
            texts13[i:i + tokenize_batch_size13],
            max_length = max_length,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt')
        input_ids13.append(batch13['input_ids'])
        attention_masks13.append(batch13['attention_mask'])
        print_memory_usage13(f"After batch {i}–{i + tokenize_batch_size13}")

    all_input_ids13 = torch.cat(input_ids13, dim = 0)
    all_attention_masks13 = torch.cat(attention_masks13, dim = 0)

    print_memory_usage13("End batch_tokenize13")
    return all_input_ids13, all_attention_masks13

def sample_10_percent_df13(input_df13):
    _, sampled_df13 = train_test_split(input_df13, test_size = 0.10,
                                     stratify = input_df13['sentiment_scoreR3'], random_state = np.random.randint(10000))
    return sampled_df13.reset_index(drop = True)

train_df_small13 = sample_10_percent_df13(train_df13)
val_df_small13 = sample_10_percent_df13(val_df13)
test_df_small13 = sample_10_percent_df13(test_df13)

train_seq13, train_mask13 = batch_tokenize13(train_df_small13['combined_text'].tolist(), tokenizer13)
val_seq13, val_mask13 = batch_tokenize13(val_df_small13['combined_text'].tolist(), tokenizer13)
test_seq13, test_mask13 = batch_tokenize13(test_df_small13['combined_text'].tolist(), tokenizer13)

print_memory_usage13("Before label tensor conversion")
train_y13 = torch.tensor(train_df_small13['sentiment_scoreR3'].tolist())
val_y13 = torch.tensor(val_df_small13['sentiment_scoreR3'].tolist())
test_y13 = torch.tensor(test_df_small13['sentiment_scoreR3'].tolist())
print_memory_usage13("After label tensor conversion")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

batch_size13 = 16

train_data13 = TensorDataset(train_seq13, train_mask13, train_y13)
val_data13   = TensorDataset(val_seq13, val_mask13, val_y13)
test_data13  = TensorDataset(test_seq13, test_mask13, test_y13)

train_sampler13 = RandomSampler(train_data13)
val_sampler13   = SequentialSampler(val_data13)
test_sampler13  = SequentialSampler(test_data13)

train_dataloader13 = DataLoader(train_data13, sampler = train_sampler13, batch_size = batch_size13)
val_dataloader13   = DataLoader(val_data13, sampler = val_sampler13, batch_size = batch_size13)
test_dataloader13  = DataLoader(test_data13, sampler = test_sampler13, batch_size = batch_size13)

for param in bert13.parameters():
    param.requires_grad = False

In [ ]:
import torch.nn as nn

class BERT_Arch13(nn.Module):  
    def __init__(self, bert13):
        super(BERT_Arch13, self).__init__()

        self.bert = bert13
        self.dropout = nn.Dropout(0.1)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(768, 512)
        self.fc2 = nn.Linear(512, 2)

    def forward(self, sent_id, mask):
        outputs = self.bert(sent_id, attention_mask = mask, return_dict = True)
        cls_hs = outputs.last_hidden_state[:, 0, :] 

        x = self.fc1(cls_hs)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x

model13 = BERT_Arch13(bert13)
model13 = model13.to(device)

In [12]:
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight

optimizer13 = AdamW(model13.parameters(), lr = 2e-5, weight_decay = 0.01)

class_weights13 = compute_class_weight(class_weight = 'balanced', classes = np.unique(train_df_small13['sentiment_scoreR3']),
                                       y = train_df_small13['sentiment_scoreR3'])

weights13 = torch.tensor(class_weights13, dtype = torch.float).to(device)

cross_entropy13 = nn.CrossEntropyLoss(weight = weights13)

epochs13 = 5

In [ ]:
def train13():
    model13.train()
    total_loss13, total_accuracy13 = 0, 0
    total_preds13 = []

    for step, batch in enumerate(train_dataloader13):
        if step % 50 == 0 and not step == 0:
            print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader13)))

        batch = [r.to(device) for r in batch]
        sent_id, mask, labels = batch

        model13.zero_grad()

        preds13 = model13(sent_id, mask)
        loss13 = cross_entropy13(preds13, labels)

        total_loss13 += loss13.item()

        loss13.backward()
        torch.nn.utils.clip_grad_norm_(model13.parameters(), 1.0)
        optimizer13.step()

        preds13 = preds13.detach().cpu().numpy()
        total_preds13.append(preds13)

    avg_loss13 = total_loss13 / len(train_dataloader13)

    if len(total_preds13) == 0:
        print("Warning: train13() produced no predictions. Check your training dataloader.")

    total_preds13 = np.concatenate(total_preds13, axis = 0)

    return avg_loss13, total_preds13

In [14]:
import datetime
import time

def format_time13(elapsed):
    return str(datetime.timedelta(seconds = int(round(elapsed))))

def evaluate13():
    print("\nEvaluating...")
    t0 = time.time()

    model13.eval()
    total_loss13, total_accuracy13 = 0, 0
    total_preds13 = []

    for step, batch in enumerate(val_dataloader13):
        if step % 50 == 0 and not step == 0:
            elapsed = format_time13(time.time() - t0)
            print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader13)))

        batch = [t.to(device) for t in batch]
        sent_id, mask, labels = batch

        with torch.no_grad():
            preds13 = model13(sent_id, mask)
            loss13 = cross_entropy13(preds13, labels)

            total_loss13 += loss13.item()
            preds13 = preds13.detach().cpu().numpy()
            total_preds13.append(preds13)

    avg_loss13 = total_loss13 / len(val_dataloader13)

    if len(total_preds13) == 0:
        print("Warning: evaluate13() produced no predictions. Check your validation dataloader.")

    total_preds13 = np.concatenate(total_preds13, axis = 0)

    return avg_loss13, total_preds13

In [ ]:
best_valid_loss13 = float('inf')

epochs13 = 5

patience13 = 2
patience_counter13 = 0

train_losses13 = []
valid_losses13 = []

for epoch in range(epochs13):
    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs13))
    print_memory_usage13(f"Start of Epoch {epoch + 1}")

    print_memory_usage13("Before training")
    train_loss13, _ = train13()
    print_memory_usage13("After training")

    print_memory_usage13("Before validation")
    valid_loss13, _ = evaluate13()
    print_memory_usage13("After validation")

    if valid_loss13 < best_valid_loss13:
        best_valid_loss13 = valid_loss13
        patience_counter13 = 0
        print_memory_usage13("Before saving model weights")
        torch.save(model13.state_dict(), 'DistilBERT_bestweights13.pt')
        print_memory_usage13("After saving model weights")
    else:
        patience_counter13 += 1
        if patience_counter13 >= patience13:
            print(f"\nEarly stopping triggered after {epoch + 1} epochs.")
            break

    train_losses13.append(train_loss13)
    valid_losses13.append(valid_loss13)

    print(f'\nTraining Loss: {train_loss13:.3f}')
    print(f'Validation Loss: {valid_loss13:.3f}')
    print_memory_usage13(f"End of Epoch {epoch + 1}")


 Epoch 1 / 5
[Start of Epoch 1] Memory usage: 1.31 GB
[Before training] Memory usage: 1.31 GB
  Batch    50  of  1,243.
  Batch   100  of  1,243.
  Batch   150  of  1,243.
  Batch   200  of  1,243.
  Batch   250  of  1,243.
  Batch   300  of  1,243.
  Batch   350  of  1,243.
  Batch   400  of  1,243.
  Batch   450  of  1,243.
  Batch   500  of  1,243.
  Batch   550  of  1,243.
  Batch   600  of  1,243.
  Batch   650  of  1,243.
  Batch   700  of  1,243.
  Batch   750  of  1,243.
  Batch   800  of  1,243.
  Batch   850  of  1,243.
  Batch   900  of  1,243.
  Batch   950  of  1,243.
  Batch 1,000  of  1,243.
  Batch 1,050  of  1,243.
  Batch 1,100  of  1,243.
  Batch 1,150  of  1,243.
  Batch 1,200  of  1,243.
[After training] Memory usage: 0.56 GB
[Before validation] Memory usage: 0.56 GB

Evaluating...
  Batch    50  of    156.
  Batch   100  of    156.
  Batch   150  of    156.
[After validation] Memory usage: 0.50 GB
[Before saving model weights] Memory usage: 0.50 GB
[After saving 

In [ ]:
from sklearn.metrics import (classification_report, accuracy_score, 
                             confusion_matrix, roc_auc_score, roc_curve)

from sklearn.preprocessing import label_binarize
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
import numpy as np

all_metrics13 = []          
all_fpr_tpr_auc13 = []      

def get_predictions13(model13, dataloader13):
    model13.eval()
    all_preds13, all_labels13 = [], []

    for batch in dataloader13:
        batch = [t.to(device) for t in batch]
        sent_id, mask, labels = batch

        with torch.no_grad():
            logits13 = model13(sent_id, mask)
            preds13 = torch.argmax(logits13, dim = 1)

        all_preds13.extend(preds13.cpu().numpy())
        all_labels13.extend(labels.cpu().numpy())

    return all_labels13, all_preds13

def get_pred_probs13(model13, dataloader13):
    model13.eval()
    all_probs13, all_labels13 = [], []

    for batch in dataloader13:
        batch = [t.to(device) for t in batch]
        sent_id, mask, labels = batch

        with torch.no_grad():
            logits13 = model13(sent_id, mask)
            probs13 = torch.softmax(logits13, dim = 1)

        all_probs13.extend(probs13.cpu().numpy())
        all_labels13.extend(labels.cpu().numpy())

    return np.array(all_labels13), np.array(all_probs13)

train_true_labels13, train_preds13 = get_predictions13(model13, train_dataloader13)
print("Training Classification Report:")
print(classification_report(train_true_labels13, train_preds13))
print("Training Accuracy:", accuracy_score(train_true_labels13, train_preds13))

val_true_labels13, val_preds13 = get_predictions13(model13, val_dataloader13)
print("\nValidation Classification Report:")
print(classification_report(val_true_labels13, val_preds13))
print("Validation Accuracy:", accuracy_score(val_true_labels13, val_preds13))

test_true_labels13, test_preds13 = get_predictions13(model13, test_dataloader13)
print("\nTest Classification Report:")
print(classification_report(test_true_labels13, test_preds13))
print("Test Accuracy:", accuracy_score(test_true_labels13, test_preds13))

Training Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.75      0.65      5547
           1       0.89      0.78      0.83     14326

    accuracy                           0.77     19873
   macro avg       0.73      0.76      0.74     19873
weighted avg       0.80      0.77      0.78     19873

Training Accuracy: 0.77240477029135

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.59      0.77      0.67       693
           1       0.90      0.79      0.84      1791

    accuracy                           0.79      2484
   macro avg       0.74      0.78      0.76      2484
weighted avg       0.81      0.79      0.79      2484

Validation Accuracy: 0.785829307568438

Test Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.76      0.65       694
           1       0.89      0.77      0.83      1791

    accurac

In [ ]:
import pandas as pd
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_curve, roc_auc_score

train_acc13 = accuracy_score(train_true_labels13, train_preds13)
val_acc13 = accuracy_score(val_true_labels13, val_preds13)
test_acc13 = accuracy_score(test_true_labels13, test_preds13)

conf_matrix_train13 = confusion_matrix(train_true_labels13, train_preds13)
conf_matrix_val13 = confusion_matrix(val_true_labels13, val_preds13)
conf_matrix_test13 = confusion_matrix(test_true_labels13, test_preds13)

def print_conf_matrix13(cm13, title13):
    print(f"\n{title13} Confusion Matrix - Model 1 Round 3 (13):")
    df13 = pd.DataFrame(cm13,
                        index = ["True Non-Positive", "True Positive"],
                        columns = ["Predicted Non-Positive", "Predicted Positive"])
    print(df13)

print_conf_matrix13(conf_matrix_train13, "Training")
print_conf_matrix13(conf_matrix_val13, "Validation")
print_conf_matrix13(conf_matrix_test13, "Test")

train_probs_true13, train_probs13 = get_pred_probs13(model13, train_dataloader13)
val_probs_true13, val_probs13 = get_pred_probs13(model13, val_dataloader13)
test_probs_true13, test_probs13 = get_pred_probs13(model13, test_dataloader13)

def get_roc_data13(y_true13, y_probs13, set_name13, model_round13 = '13'):
    if y_probs13.ndim == 2 and y_probs13.shape[1] == 2:
        y_probs13 = y_probs13[:, 1]  
    elif y_probs13.ndim > 1 and y_probs13.shape[1] == 1:
        y_probs13 = y_probs13.ravel()
        
    y_bin13 = label_binarize(y_true13, classes = [0, 1]).ravel()
    fpr13, tpr13, _ = roc_curve(y_bin13, y_probs13)
    auc13 = roc_auc_score(y_bin13, y_probs13)

    return [{
        'model_round': model_round13,
        'set': set_name13,
        'class': 1,
        'fpr': fpr13,
        'tpr': tpr13,
        'auc': auc13}]

roc_train13 = get_roc_data13(train_probs_true13, train_probs13, set_name13 = 'train')
roc_val13 = get_roc_data13(val_probs_true13, val_probs13, set_name13 = 'val')
roc_test13 = get_roc_data13(test_probs_true13, test_probs13, set_name13 = 'test')

round_metrics13 = {
    'model_round': '13',
    'train_accuracy': train_acc13,
    'val_accuracy': val_acc13,
    'test_accuracy': test_acc13,
    'train_report': classification_report(train_true_labels13, train_preds13, output_dict = True),
    'val_report': classification_report(val_true_labels13, val_preds13, output_dict = True),
    'test_report': classification_report(test_true_labels13, test_preds13, output_dict = True),
    'conf_matrix_train13': conf_matrix_train13,
    'conf_matrix_val13': conf_matrix_val13,
    'conf_matrix_test13': conf_matrix_test13}

all_metrics13.append(round_metrics13)

all_fpr_tpr_auc13.append({
    'model_round': '13',
    'roc_train13': roc_train13,
    'roc_val13': roc_val13,
    'roc_test13': roc_test13})


Training Confusion Matrix - Model 1 Round 3 (13):
                   Predicted Non-Positive  Predicted Positive
True Non-Positive                    4145                1402
True Positive                        3121               11205

Validation Confusion Matrix - Model 1 Round 3 (13):
                   Predicted Non-Positive  Predicted Positive
True Non-Positive                     537                 156
True Positive                         376                1415

Test Confusion Matrix - Model 1 Round 3 (13):
                   Predicted Non-Positive  Predicted Positive
True Non-Positive                     525                 169
True Positive                         405                1386


In [ ]:
from pprint import pprint

print("Stored Metrics for Model 1 Round 3")
pprint(all_metrics13[-1])  

==== Stored Metrics for Model 1 Round 3 ====
{'conf_matrix_test13': array([[ 525,  169],
       [ 405, 1386]], dtype=int64),
 'conf_matrix_train13': array([[ 4145,  1402],
       [ 3121, 11205]], dtype=int64),
 'conf_matrix_val13': array([[ 537,  156],
       [ 376, 1415]], dtype=int64),
 'model_round': '13',
 'test_accuracy': 0.7690140845070422,
 'test_report': {'0': {'f1-score': 0.646551724137931,
                       'precision': 0.5645161290322581,
                       'recall': 0.7564841498559077,
                       'support': 694.0},
                 '1': {'f1-score': 0.8284518828451883,
                       'precision': 0.8913183279742766,
                       'recall': 0.7738693467336684,
                       'support': 1791.0},
                 'accuracy': 0.7690140845070422,
                 'macro avg': {'f1-score': 0.7375018034915597,
                               'precision': 0.7279172285032673,
                               'recall': 0.765176748294788,
   

In [ ]:
print("\nAccuracies:")
print("Training:", all_metrics13[-1]['train_accuracy'])
print("Validation:  ", all_metrics13[-1]['val_accuracy'])
print("Test: ", all_metrics13[-1]['test_accuracy'])

In [23]:
print("\nTest Confusion Matrix:")
print(all_metrics13[-1]['conf_matrix_test13'])


Test Confusion Matrix:
[[ 525  169]
 [ 405 1386]]


In [24]:
print("\nTraining Confusion Matrix:")
print(all_metrics13[-1]['conf_matrix_train13'])


Training Confusion Matrix:
[[ 4145  1402]
 [ 3121 11205]]


In [25]:
print("\nValidation Confusion Matrix:")
print(all_metrics13[-1]['conf_matrix_val13'])


Validation Confusion Matrix:
[[ 537  156]
 [ 376 1415]]


In [27]:
print("\nAUC Scores by Class for Test Set:")
for entry in all_fpr_tpr_auc13[-1]['roc_test13']:
    print(f"Class {entry['class']} - AUC: {entry['auc']:.4f}")


AUC Scores by Class for Test Set:
Class 1 - AUC: 0.8474


In [28]:
print("\nAUC Scores by Class for Training Set:")
for entry in all_fpr_tpr_auc13[-1]['roc_train13']:
    print(f"Class {entry['class']} - AUC: {entry['auc']:.4f}")


AUC Scores by Class for Training Set:
Class 1 - AUC: 0.8466


In [29]:
print("\nAUC Scores by Class for Validation Set:")
for entry in all_fpr_tpr_auc13[-1]['roc_val13']:
    print(f"Class {entry['class']} - AUC: {entry['auc']:.4f}")


AUC Scores by Class for Validation Set:
Class 1 - AUC: 0.8599


In [31]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['font.family'] = 'Georgia'

def roc_data13(class_id, roc_data13):
    for entry13 in roc_data13: 
        if entry13['class'] == class_id:
            return entry13['fpr'], entry13['tpr'], entry13['auc']
    return None, None, None

roc_entry13 = all_fpr_tpr_auc13[-1]
roc_train13 = roc_entry13['roc_train13']
roc_val13 = roc_entry13['roc_val13']
roc_test13 = roc_entry13['roc_test13']

In [ ]:
fpr_train13, tpr_train13, auc_train13 = roc_data13(1, roc_train13)
fpr_val13, tpr_val13, auc_val13 = roc_data13(1, roc_val13)
fpr_test13, tpr_test13, auc_test13 = roc_data13(1, roc_test13)

fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(fpr_train13, tpr_train13, label = f'Training (AUC = {auc_train13:.2f})', linewidth = 2)
ax.plot(fpr_val13, tpr_val13, label = f'Validation (AUC = {auc_val13:.2f})', linewidth = 2)
ax.plot(fpr_test13, tpr_test13, label = f'Test (AUC = {auc_test13:.2f})', linewidth = 2)
ax.plot([0, 1], [0, 1], 'k--', linewidth = 1)

ax.set_title("DistilBERT (Round 3) - ROC Curve for Class 1 (Positive)", fontweight = 'bold', fontsize = 14)
ax.set_xlabel("False Positive Rate (FPR)", fontsize = 12, fontweight = 'bold')
ax.set_ylabel("True Positive Rate (TPR)", fontsize = 12, fontweight = 'bold')
ax.tick_params(axis = 'both', labelsize = 10)
ax.grid(True, linestyle = '--', linewidth = 0.5, alpha = 0.7)
ax.legend(fontsize = 10)

plt.tight_layout()
plt.show()

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler

tokens_train_small13 = tokenizer13.batch_encode_plus(train_df_small13['combined_text'].tolist(),
                                                     max_length = 128,
                                                     padding = 'max_length',
                                                     truncation = True, 
                                                     return_tensors = 'pt')

train_small_seq13 = tokens_train_small13['input_ids']
train_small_mask13 = tokens_train_small13['attention_mask']

train_small_data13 = TensorDataset(train_small_seq13, train_small_mask13)
train_small_dataloader13 = DataLoader(train_small_data13,
                                      sampler = SequentialSampler(train_small_data13),
                                      batch_size = 32)

model13.eval()
train_small_pred_labels13 = []
train_small_probs13 = []

for batch in train_small_dataloader13:
    batch = [t.to(device) for t in batch]
    sent_id, mask = batch

    with torch.no_grad():
        logits13 = model13(sent_id, mask)
        probs13 = torch.softmax(logits13, dim = 1)
        preds13 = torch.argmax(probs13, dim = 1)

    train_small_pred_labels13.extend(preds13.cpu().numpy())
    train_small_probs13.extend(probs13.cpu().numpy())

train_small_pred_labels13 = np.array(train_small_pred_labels13)
train_small_probs13 = np.array(train_small_probs13)

bert1_round3_df = train_df_small13.reset_index(drop = True).copy()
bert1_round3_df['bert_pred_label13'] = train_small_pred_labels13
bert1_round3_df['bert_pred_prob_nonpos13'] = train_small_probs13[:, 0]
bert1_round3_df['bert_pred_prob_pos13'] = train_small_probs13[:, 1]

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

OMAI_distilbert_round3_output = bert1_round3_df.copy(deep = True)
OMAI_distilbert_round3_output.iloc[0:1]
print(OMAI_distilbert_round3_output.shape)

In [ ]:
OMAI_distilbert_round3_output.to_csv('OMAI - Results - DistilBERT (Round 3) (Part 1).csv')